## Environment Setup and Data Loading
Loading the Pakistan used car price dataset and verifying the environment is ready. A fixed random seed
is set for reproducibility throughout the project.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility: fixed seed used throughout the project
RANDOM_STATE = 42

# Load the dataset — replace with your actual filename if different
df = pd.read_csv("../Dataset/used_cars_Pakistan.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Rows: 86120
Columns: 9


,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs
0,0,Suzuki,Alto,Lahore,2019,45744,660.0,NaN,3550000
1,1,Suzuki,Wagon,Lahore,2019,17583,660.0,NaN,3850000
2,2,Suzuki,Wagon,Lahore,2019,64085,660.0,NaN,3890000
3,3,Suzuki,Wagon,Lahore,2019,71281,660.0,NaN,3990000
4,4,Toyota,Aqua,Lahore,2020,19950,1500.0,NaN,3990000


## Data Structure
Checking column types and non-null counts. The dataset has 9 raw columns: 4 numeric, 3 categorical (text),
plus Battery which is mostly empty.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 86120 entries, 0 to 86119
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           86120 non-null  int64  
 1   make                 86120 non-null  str    
 2   model                86120 non-null  str    
 3   city                 86120 non-null  str    
 4   year                 86120 non-null  int64  
 5   mileage              86120 non-null  int64  
 6   Engine_displacement  85931 non-null  float64
 7   Battery              189 non-null    float64
 8   Price_Rs             86120 non-null  int64  
dtypes: float64(2), int64(4), str(3)
memory usage: 5.9 MB


## Data Quality and Basic Statistics
Checking missing values, duplicate rows, and basic statistics to identify data quality issues before
modeling.

**Findings:** Engine_displacement has 189 missing values; Battery is populated for only 189 of 86,120 rows
(0.2%), suggesting it marks electric/hybrid vehicles specifically. Price_Rs has a minimum of
-2,147,483,648, exactly the minimum value for a 32-bit integer, indicating an integer overflow bug.
year ranges from 1944 to 2023, and Engine_displacement reaches an implausible 15,000cc, both suggesting
a small number of erroneous or extreme entries.

In [3]:
df.describe()

,Unnamed: 0,year,mileage,Engine_displacement,Battery,Price_Rs
count,86120.000000,86120.000000,86120.000000,85931.000000,189.000000,8.612000e+04
mean,43059.500000,2012.669995,88789.209963,1388.138146,119.653439,4.173561e+06
std,24860.846928,8.862183,87932.720095,766.087270,207.891986,9.717305e+06
min,0.000000,1944.000000,1.000000,0.000000,3.000000,-2.147484e+09
25%,21529.750000,2007.000000,31000.000000,1000.000000,60.000000,1.500000e+06
50%,43059.500000,2015.000000,78000.000000,1300.000000,80.000000,2.800000e+06
75%,64589.250000,2020.000000,121000.000000,1600.000000,101.000000,4.750000e+06
max,86119.000000,2023.000000,1000000.000000,15000.000000,2000.000000,3.015000e+08


## Investigating Data Quality Issues
Quantifying the scope of each quality issue found in the previous step: price overflow, suspicious years,
and engine displacement anomalies.

**Findings:** 1 row has a corrupted negative price (32-bit integer overflow) and 112 rows have
suspiciously low prices (<50,000 PKR), both likely errors. Years before 1990 account for 2,293 rows and
before 2000 for 7,400 rows (8.6%), but the year-by-year counts show a smooth, gradually increasing tail
rather than random spikes, suggesting these are genuine older vehicles still circulating in Pakistan's
used car market rather than data errors. For engine displacement, cross-checking against the Battery
column revealed that all 189 rows with Engine_displacement == NaN have a Battery value (confirming these
are genuine electric vehicles with no combustion engine), while the 2,993 rows with Engine_displacement
== 0 have no Battery value, indicating 0 is a placeholder for missing data on non-EV cars rather than a
real measurement. 173 rows have an implausible engine size above 5,000cc, likely data entry errors.

In [4]:
#checking any overflow issues in the Price_Rs column
print("Negative prices:", (df["Price_Rs"] < 0).sum())
print("Very low prices (under 50,000 PKR):", (df["Price_Rs"] < 50000).sum())
#checking suspicious year values in the Year column
print("Year < 1990:", (df["year"] < 1990).sum())
print("Year < 2000:", (df["year"] < 2000).sum())
#checking any extreme engine displacment
print("Engine_displacement > 5000:", (df["Engine_displacement"] > 5000).sum())
print("Engine_displacement == 0:", (df["Engine_displacement"] == 0).sum())

Negative prices: 1
Very low prices (under 50,000 PKR): 112
Year < 1990: 2293
Year < 2000: 7400
Engine_displacement > 5000: 173
Engine_displacement == 0: 2993


In [5]:
df["year"].value_counts().sort_index().head(20)

year
1944     2
1946     1
1950     2
1951     1
1952     6
1960     9
1961     5
1962     3
1963     4
1964     5
1965     6
1966     3
1967    10
1968     8
1969    16
1970     6
1971     5
1972     8
1973    17
1974    23
Name: count, dtype: int64

In [6]:
# How many rows have engine == 0 AND a battery value (likely real EVs)?
print("Engine 0 AND Battery not null:", ((df["Engine_displacement"] == 0) & (df["Battery"].notna())).sum())

# How many rows have engine == 0 but NO battery value (likely missing data, not EVs)?
print("Engine 0 AND Battery is null:", ((df["Engine_displacement"] == 0) & (df["Battery"].isna())).sum())

print("Battery not null AND Engine_displacement is NaN:", ((df["Battery"].notna()) & (df["Engine_displacement"].isna())).sum())

Engine 0 AND Battery not null: 0
Engine 0 AND Battery is null: 2993
Battery not null AND Engine_displacement is NaN: 189


## Investigating Low-Price Rows
Inspecting the 112 rows with Price_Rs under 50,000 PKR to check for a pattern before deciding how to
handle them.

**Findings:** 111 of the 112 rows have Price_Rs exactly equal to 0, spread across many different makes
(Suzuki, Toyota, Honda, Mercedes, KIA, Nissan, and others), ruling out a brand-specific scraping bug.
This indicates 0 is a placeholder for "price not listed," consistent with other placeholder values found
elsewhere in this dataset (e.g. Engine_displacement == 0). Since Price_Rs is the target variable, these
rows cannot be imputed like a missing feature, they must be dropped, along with the single row affected
by the integer overflow bug.

In [7]:
#checking if the prices < 50,000 are valid or not 
df[df["Price_Rs"] < 50000].sort_values("Price_Rs").head(20)

,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs
785,785,Suzuki,Khyber,Karachi,1987,252858,1000.0,NaN,-2147483648
86,86,Haval,Jolion,Islamabad,2023,15,1500.0,NaN,0
123,123,Honda,N,Multan,2018,63000,660.0,NaN,0
197,197,Mitsubishi,EK,Multan,2009,90000,660.0,NaN,0
1011,1011,Suzuki,Alto,Lahore,2023,27,660.0,NaN,0
2565,2565,Toyota,Prado,Islamabad,2016,55000,2700.0,NaN,0
4008,4008,Honda,Vezel,Rawalpindi,2021,15000,1500.0,NaN,0
4009,4009,Toyota,Land,Islamabad,1986,100000,3500.0,NaN,0
4933,4933,Toyota,C-HR,Rawalpindi,2018,51350,1800.0,NaN,0
6413,6413,KIA,Sportage,Islamabad,2023,25,1999.0,NaN,0


In [8]:
#checking 20 lowest priced cars
df[df["Price_Rs"] < 50000]["make"].value_counts()

make
Toyota        49
Honda         12
Suzuki        10
Hyundai       10
Haval          5
Mercedes       4
Nissan         4
MG             3
Audi           3
Mazda          3
Mitsubishi     2
KIA            2
BAIC           2
Ford           1
BMW            1
Daihatsu       1
Name: count, dtype: int64

## Removing Invalid Target Rows
Dropping rows where Price_Rs is a placeholder (0) or corrupted (negative, from the integer overflow bug),
since the target variable cannot be imputed.

In [9]:
#for bad rows
bad_price_mask = df["Price_Rs"] < 50000
print("Rows to drop:", bad_price_mask.sum())

df = df[~bad_price_mask].reset_index(drop=True)
print("Rows remaining:", df.shape[0])

Rows to drop: 112
Rows remaining: 86008


In [10]:
df[df["Engine_displacement"] > 5000].sort_values("Engine_displacement", ascending=False).head(20)

,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs
68000,68111,Toyota,Corolla,Lahore,2021,85000,15000.0,NaN,4150000
83102,83214,Honda,Civic,Bahawalpur,1997,200000,15000.0,NaN,1000000
58351,58455,Suzuki,Khyber,Islamabad,1998,5555,15000.0,NaN,500000
52856,52960,Honda,Civic,Sialkot,2005,216611,15000.0,NaN,1500000
25327,25419,Honda,Civic,Islamabad,1995,10000,15000.0,NaN,750000
63523,63631,Prince,Pearl,Multan,2022,7000,15000.0,NaN,1500000
56069,56173,Toyota,Corolla,Swat,2010,1,15000.0,NaN,3000000
21724,21816,Honda,Freed,Gujranwala,2014,85000,14983.0,NaN,4050000
78626,78738,Suzuki,Swift,Lahore,2012,220000,14000.0,NaN,1850000
71517,71629,Suzuki,Margalla,Charsadda,1994,90000,13000.0,NaN,570000


## Cleaning Engine Displacement
Converting invalid Engine_displacement values (0, used as a missing-data placeholder, and values above
5000cc, which are implausible for passenger cars) to NaN, so they can be properly imputed later rather
than treated as real measurements.

**Findings:** 3,166 values were converted (2,993 zeros + 173 extreme outliers), bringing total missing
Engine_displacement to 3,354 (3.9% of the remaining 86,008 rows). This count is one less than the expected
3,355 because one of the rows dropped earlier for an invalid price was itself a genuine electric vehicle
with an already-missing Engine_displacement value.

In [11]:
# Treat 0 and implausibly large engine values as missing, not real measurements
invalid_engine_mask = (df["Engine_displacement"] == 0) | (df["Engine_displacement"] > 5000)
print("Rows with invalid engine displacement:", invalid_engine_mask.sum())

df.loc[invalid_engine_mask, "Engine_displacement"] = np.nan
print("Total missing values in Engine_displacement after cleaning:", df["Engine_displacement"].isna().sum())

Rows with invalid engine displacement: 3166
Total missing values in Engine_displacement after cleaning: 3354


## Feature Engineering: is_electric
Creating a binary flag indicating whether a listing is an electric vehicle, based on whether Battery has
a recorded value.

**Finding:** 188 of 86,008 listings (0.2%) are electric vehicles.

In [12]:
# Making a new column (is_electric) to indicate whether the car is electric or not based on the Battery column
df["is_electric"] = df["Battery"].notna().astype(int)
print("Number of electric cars:", df["is_electric"].sum())

Number of electric cars: 188


## Feature Engineering: car_age
Creating car_age as a reference year (2024) minus the manufacturing year, since age is more directly
interpretable for pricing than raw year.

**Finding:** Ages range from 1 to 80 years (median 9), right-skewed as expected, with no negative values
confirming no data errors in this calculation.

In [13]:
#new column for car age
df["car_age"] = 2024 - df["year"]
df["car_age"].describe()

count    86008.000000
mean        11.338620
std          8.862557
min          1.000000
25%          4.000000
50%          9.000000
75%         17.000000
max         80.000000
Name: car_age, dtype: float64

In [14]:
#safety check for for any car ages equal to 0 (zero)
print("Rows with car_age == 0:", (df["car_age"] == 0).sum())

Rows with car_age == 0: 0


In [15]:
#new column for mileage per year
df["mileage_per_year"] = df["mileage"] / df["car_age"]
df["mileage_per_year"].describe()

count     86008.000000
mean       9110.357105
std        7385.237781
min           0.015873
25%        5000.000000
50%        8333.333333
75%       12000.000000
max      249999.750000
Name: mileage_per_year, dtype: float64

In [16]:
df.sort_values("mileage_per_year", ascending=False).head(10)

,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs,is_electric,car_age,mileage_per_year
54960,55064,Suzuki,Cultus,Faisalabad,2020,999999,1000.0,NaN,2875000,0,4,249999.750000
49393,49495,Toyota,Coaster,Faisalabad,2023,234000,4000.0,NaN,12000000,0,1,234000.000000
44094,44192,Honda,BR-V,Sadiqabad,2021,580000,NaN,NaN,5200000,0,3,193333.333333
3444,3450,Mitsubishi,Minica,Lahore,2023,185000,660.0,NaN,2800000,0,1,185000.000000
24370,24462,Honda,City,Lahore,2019,900000,1500.0,NaN,4300000,0,5,180000.000000
49755,49857,Toyota,Corolla,Chishtian,2019,890000,1300.0,NaN,4450000,0,5,178000.000000
5367,5376,Honda,BR-V,Karachi,2019,880000,1500.0,NaN,4230000,0,5,176000.000000
78898,79010,Honda,BR-V,Faisalabad,2021,520000,1500.0,NaN,5400000,0,3,173333.333333
4204,4212,Toyota,Corolla,Wah cantt,2019,840000,1300.0,NaN,4485000,0,5,168000.000000
47830,47932,Suzuki,Mehran,Chowk munda,2018,1000000,NaN,NaN,1250000,0,6,166666.666667


In [17]:
#check the raw mileage column's distribution directly to decide a sensible cutoff
df["mileage"].describe()
print("\nMileage > 500,000:", (df["mileage"] > 500000).sum())
print("Mileage > 300,000:", (df["mileage"] > 300000).sum())


Mileage > 500,000: 494
Mileage > 300,000: 1243


In [18]:
print("mileage_per_year > 100,000:", (df["mileage_per_year"] > 100000).sum())
print("mileage_per_year > 50,000:", (df["mileage_per_year"] > 50000).sum())

mileage_per_year > 100,000: 67
mileage_per_year > 50,000: 238


In [19]:
# 1. Mask: rows where mileage_per_year is above 100,000
bad_mileage_mask = df["mileage_per_year"] > 100000

# 2. Check the count before changing anything
print("Rows to fix:", bad_mileage_mask.sum())

# 3. Blank out mileage for those rows only
df.loc[bad_mileage_mask, "mileage"] = np.nan

# 4. Recalculate mileage_per_year from the updated mileage
df["mileage_per_year"] = df["mileage"] / df["car_age"]

# 5. Verify
print("Missing mileage:", df["mileage"].isna().sum())
print("Rows still above 100,000:", (df["mileage_per_year"] > 100000).sum())

Rows to fix: 67
Missing mileage: 67
Rows still above 100,000: 0


In [20]:
df.columns.tolist()

['Unnamed: 0',
 'make',
 'model',
 'city',
 'year',
 'mileage',
 'Engine_displacement',
 'Battery',
 'Price_Rs',
 'is_electric',
 'car_age',
 'mileage_per_year']

## Duplicate Check
Checking for repeated listings by comparing every column except the row-number column (Unnamed: 0, which
is unique per row and would hide all duplicates).

**Findings:** 7,762 rows (9.02%) are exact repeats of an earlier row. Inspecting them shows the same
make, model, city, year, odometer reading, engine size and price, with some listings appearing up to three
times, which indicates the same advertisement was scraped multiple times rather than separate cars.
Keeping them would overweight those listings and could leak identical rows into both the training and
test sets, inflating performance.

In [21]:
cols_to_check = [c for c in df.columns if c != "Unnamed: 0"]

#count duplicate rows
dup_count = df.duplicated(subset = cols_to_check).sum()
print("Duplicate rows:", dup_count)
print(f"Share of dataset:{dup_count / df.shape[0]:.2%}")

df[df.duplicated(subset=cols_to_check, keep=False)].sort_values(
    ["make", "model", "year", "mileage"]
).head(10)

Duplicate rows: 7762
Share of dataset:9.02%


,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs,is_electric,car_age,mileage_per_year
14294,14335,Audi,A3,Gujrat,2014,98000.0,1400.0,NaN,6300000,0,10,9800.000000
32564,32658,Audi,A3,Gujrat,2014,98000.0,1400.0,NaN,6300000,0,10,9800.000000
60052,60156,Audi,A3,Gujrat,2014,98000.0,1400.0,NaN,6300000,0,10,9800.000000
1082,1087,Audi,A3,Lahore,2015,45954.0,1800.0,NaN,6850000,0,9,5106.000000
1582,1587,Audi,A3,Lahore,2015,45954.0,1800.0,NaN,6850000,0,9,5106.000000
17717,17780,Audi,A3,Islamabad,2015,72000.0,1800.0,NaN,6500000,0,9,8000.000000
49251,49353,Audi,A3,Islamabad,2015,72000.0,1800.0,NaN,6500000,0,9,8000.000000
11241,11268,Audi,A3,Lahore,2015,104000.0,1800.0,NaN,6900000,0,9,11555.555556
23627,23719,Audi,A3,Lahore,2015,104000.0,1800.0,NaN,6900000,0,9,11555.555556
6258,6267,Audi,A3,Lahore,2015,125000.0,1800.0,NaN,6700000,0,9,13888.888889


## Removing Duplicate Listings
Dropping repeated rows (keeping the first occurrence) so each listing is counted once, then resetting the
index.

**Result:** 86,008 rows reduced to 78,246 (7,762 removed), with 0 duplicates remaining. Removing duplicates
before the train/test split also prevents identical listings from appearing in both sets.

In [22]:
rows_before = df.shape[0]

#drop duplicate rows
df = df.drop_duplicates(subset = cols_to_check, keep = "first").reset_index(drop = True)

print("Rows before:", rows_before)
print("Rows after:", df.shape[0])
print("Duplicates remaining:", df.duplicated(subset = cols_to_check).sum())

Rows before: 86008
Rows after: 78246
Duplicates remaining: 0


In [23]:
df["Price_Rs"].describe()

count    7.824600e+04
mean     3.919668e+06
std      5.841746e+06
min      1.129990e+05
25%      1.450000e+06
50%      2.680000e+06
75%      4.500000e+06
max      3.015000e+08
Name: Price_Rs, dtype: float64

In [24]:
# 20 cheapest listings remaining in the data
df.sort_values("Price_Rs").head(20)

,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs,is_electric,car_age,mileage_per_year
135,144,Suzuki,Mehran,Rawalpindi,2016,65000.0,796.0,NaN,112999,0,8,8125.000000
375,435,Toyota,Corolla,Kohat,2000,198000.0,1500.0,NaN,117000,0,24,8250.000000
58676,66027,Suzuki,FX,Sadiqabad,1987,1000000.0,800.0,NaN,120000,0,37,27027.027027
937,1148,FAW,X-PV,Haripur,2016,50000.0,1000.0,NaN,120000,0,8,6250.000000
29910,35375,Suzuki,FX,Karachi,1985,175000.0,800.0,NaN,125000,0,39,4487.179487
26401,31484,Daihatsu,Charade,Karachi,1984,9999.0,NaN,NaN,130000,0,40,249.975000
68369,76000,Suzuki,FX,Karachi,1988,65000.0,800.0,NaN,130000,0,36,1805.555556
13953,17050,Suzuki,FX,Karachi,1987,100000.0,800.0,NaN,135000,0,37,2702.702703
28959,34326,Suzuki,FX,Karachi,1982,100000.0,800.0,NaN,145000,0,42,2380.952381
7230,9214,Suzuki,FX,Karachi,1985,999.0,800.0,NaN,150000,0,39,25.615385


In [25]:
# How many listings fall under each price threshold
for threshold in [200000, 300000, 500000]:
    count = (df["Price_Rs"] < threshold).sum()
    print(f"Price under {threshold:,}: {count}")

Price under 200,000: 88
Price under 300,000: 568
Price under 500,000: 2498


In [26]:
# Recent cars (2005 or newer) priced under 500,000 PKR
recent_cheap_mask = (df["year"] >= 2005) & (df["Price_Rs"] < 500000)
print("Recent cars under 500,000 PKR:", recent_cheap_mask.sum())

# The 20 cheapest of them
df[recent_cheap_mask].sort_values("Price_Rs").head(20)

Recent cars under 500,000 PKR: 165


,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs,is_electric,car_age,mileage_per_year
135,144,Suzuki,Mehran,Rawalpindi,2016,65000.0,796.0,NaN,112999,0,8,8125.000000
937,1148,FAW,X-PV,Haripur,2016,50000.0,1000.0,NaN,120000,0,8,6250.000000
13548,16582,Changan,Karvaan,Rawalpindi,2019,46772.0,1000.0,NaN,170000,0,5,9354.400000
296,335,Mitsubishi,EK,Multan,2009,90000.0,660.0,NaN,200000,0,15,6000.000000
987,1227,Suzuki,Alto,Rawalpindi,2021,44000.0,660.0,NaN,229000,0,3,14666.666667
79,82,Honda,Civic,Faisalabad,2011,148000.0,1800.0,NaN,275000,0,13,11384.615385
20051,24289,Toyota,Fortuner,Karachi,2022,900.0,2800.0,NaN,280000,0,2,450.000000
44605,51180,Suzuki,Ravi,Lahore,2014,100000.0,800.0,NaN,300000,0,10,10000.000000
45406,52033,Sogo,Pickup,Karachi,2006,90000.0,800.0,NaN,300000,0,18,5000.000000
54490,61606,Haval,H6,Islamabad,2023,20.0,1500.0,NaN,300000,0,1,20.000000


## Removing Implausibly Cheap Recent Listings
Dropping listings from 2015 or later priced under 500,000 PKR. Recent cars at these prices (for example a
2023 Toyota Fortuner at 300,000 with 25 km on the odometer) are unlikely to be full sale prices and may
reflect down payments or installment amounts. Older cheap cars were kept, since low prices are realistic for
them.

**Result:** 24 rows removed (0.03%), leaving 78,222 rows.

In [27]:
rows_before = df.shape[0]

# Recent cars (2015 or newer) priced under 500,000 PKR
cheap_recent_mask = (df["year"] >= 2015) & (df["Price_Rs"] < 500000)
print("Rows to drop:", cheap_recent_mask.sum())

# Keep everything that does NOT match, then renumber the rows
df = df[~cheap_recent_mask].reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after:", df.shape[0])
print("Rows still matching:", ((df["year"] >= 2015) & (df["Price_Rs"] < 500000)).sum())   # expect 0

Rows to drop: 24
Rows before: 78246
Rows after: 78222
Rows still matching: 0
